---
title: "Load and Embed: SCONE Inference"
subtitle: "Load pretrained model and run inference on Kotliarov2020"
author: "Dr <strong>Pedro Henrique da Costa Avelar</strong><br> (edited by Dr Jon Cardoso-Silva)"
date: 2026
---

Demonstrates loading a pretrained SCONE model and running inference on the Kotliarov2020 single-cell dataset. We compute clustering quality metrics (ARI, AMI) against reference cell type labels and reproduce the t-SNE visualisations from Figure 3 of the paper.

⚙️ **Imports and Constants**

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import scanpy as sc
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score

import matplotlib.pyplot as plt
import seaborn as sns

import torch

from IPython.display import Markdown, display

# Notebook is in notebooks/, so project root is one level up
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import DATA_PATH, SCRNA_FNAME, SCCITE_FNAME
from nemo.models.subsetcontrastive import SubsetContrastive

# Section 1: Validate and Load Local Data

In [ ]:
# Data must already exist locally. If files are missing, run the download notebook first.
DATASETS = [
    ("rna", SCRNA_FNAME),
    ("cite", SCCITE_FNAME),
]

download_notebook_path = PROJECT_ROOT / "experiments" / "kotliarov2020" / "01_download_data.ipynb"
display(Markdown(f"**Data directory:** `{DATA_PATH}`"))
display(Markdown(f"**If data is missing, run:** `{download_notebook_path}`"))


def load_required_dataset(data_type, fname):
    """Load one required dataset and return status metadata for display."""
    fpath = DATA_PATH / fname
    exists = fpath.exists()
    adata = sc.read_h5ad(fpath) if exists else None
    return {
        "modality": data_type,
        "filename": fname,
        "path": str(fpath),
        "status": "found" if exists else "missing",
        "adata": adata,
    }


records = [load_required_dataset(data_type, fname) for data_type, fname in DATASETS]
status_df = pd.DataFrame([{k: v for k, v in rec.items() if k != "adata"} for rec in records])
display(status_df[["modality", "filename", "status", "path"]])

missing_files = [rec["path"] for rec in records if rec["status"] == "missing"]
if missing_files:
    missing_list = "\n".join([f"- {path}" for path in missing_files])
    raise FileNotFoundError(
        "Required data files are missing.\n"
        f"Data directory: {DATA_PATH}\n"
        f"Download notebook: {download_notebook_path}\n"
        f"Missing files:\n{missing_list}"
    )

data = {rec["modality"]: rec["adata"] for rec in records}
display(Markdown("All required datasets found and loaded."))

In [ ]:
for data_type, adata in data.items():
    display(Markdown(f"**{data_type}:** {adata.n_obs:,} cells x {adata.n_vars:,} features"))
    cell_types = sorted(adata.obs["cell_type"].unique())
    display(Markdown(f"- Cell types ({len(cell_types)}): {', '.join(cell_types)}"))

# Section 2: Load Pretrained Model

In [ ]:
# Path to pretrained model (in parent/pretrained/)
model_path = PROJECT_ROOT / "pretrained" / "kotliarov2020-scone-run0"

if not model_path.exists():
    raise FileNotFoundError(
        f"Pretrained model not found at {model_path}. "
        "Please train a model first using experiments/kotliarov2020/02_train_scone.py"
    )

display(Markdown(f"**Loading model from:** `{model_path}`"))

# Load the model with version=1 (required by current API)
gae = SubsetContrastive(data, version_to_load_as=1, path_to_load_from=str(model_path))
display(Markdown(f"**Model loaded.** Modalities: {', '.join(gae.modalities)}"))

# Section 3: Run Inference

In [ ]:
# Prepare tensors for inference (no subsetting — use full data)
Xs = [torch.tensor(data[m].X, dtype=torch.float32) for m in gae.modalities]
As = [
    torch.tensor(np.stack(data[m].obsp["connectivities"].nonzero()), dtype=torch.long)
    for m in gae.modalities
]

display(Markdown(f"**RNA expression:** {Xs[0].shape}"))
display(Markdown(f"**CITE protein:** {Xs[1].shape}"))

In [ ]:
# Forward pass through model to get embeddings
display(Markdown("Running inference..."))
with torch.no_grad():
    _, __, z = gae.model(Xs, As)

z_np = z.detach().cpu().numpy()

# Create AnnData object with embeddings
adz = sc.AnnData(z_np)
adz.obs = data["rna"].obs.copy()
adz.var_names = [f"dim_{i}" for i in range(z_np.shape[1])]

display(Markdown(f"**Embedding complete.** Shape: {adz.shape}"))
display(Markdown(f"**Annotations:** {', '.join(adz.obs.columns.tolist())}"))

# Section 4: Clustering Quality

In [ ]:
# Compute nearest neighbours and Leiden clustering
display(Markdown("Computing neighbourhood and Leiden clustering..."))
sc.pp.neighbors(adz, n_neighbors=15, use_rep="X", metric="cosine")
sc.tl.leiden(adz, resolution=1.0)

n_clusters = adz.obs['leiden'].nunique()
display(Markdown(f"**Leiden clustering complete.** Clusters: {n_clusters}"))

In [ ]:
# ARI and AMI against reference labels
ari_broad = adjusted_rand_score(adz.obs["cell_type"], adz.obs["leiden"])
ami_broad = adjusted_mutual_info_score(adz.obs["cell_type"], adz.obs["leiden"])

ari_fine = adjusted_rand_score(adz.obs["cluster_level3"], adz.obs["leiden"])
ami_fine = adjusted_mutual_info_score(adz.obs["cluster_level3"], adz.obs["leiden"])

summary_df = pd.DataFrame({
    "Comparison": ["vs broad cell type", "vs fine cell type"],
    "ARI": [ari_broad, ari_fine],
    "AMI": [ami_broad, ami_fine]
})

display(Markdown("**Clustering Quality Metrics:**"))
display(summary_df)

# Section 5: t-SNE Visualisation

In [ ]:
# Compute t-SNE
display(Markdown("Computing t-SNE (this may take a moment)..."))
sc.tl.tsne(adz, n_jobs=4, random_state=42)
display(Markdown("**t-SNE complete.**"))

t-SNE coloured by batch

In [ ]:
sc.pl.tsne(adz, color="batch", title="Batch", show=True)

t-SNE coloured by broad cell type

In [ ]:
sc.pl.tsne(adz, color="cell_type", title="Cell Type (Broad)", show=True)

t-SNE coloured by fine cell type

In [ ]:
sc.pl.tsne(adz, color="cluster_level3", title="Cell Type (Fine)", show=True)

# Section 6: Multi-Panel Figure (Figure 3 Reproduction)

In [ ]:
# Create a 3-panel figure matching Figure 3 of the paper
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sc.pl.tsne(adz, color="batch", ax=axes[0], show=False, title="Batch")
sc.pl.tsne(adz, color="cell_type", ax=axes[1], show=False, title="Cell Type (Broad)")
sc.pl.tsne(adz, color="cluster_level3", ax=axes[2], show=False, title="Cell Type (Fine)")

plt.tight_layout()
plt.savefig("tsne-kotliarov2020-scone-figure3.png", dpi=300, bbox_inches="tight")
display(Markdown("**Saved:** `tsne-kotliarov2020-scone-figure3.png`"))
plt.show()